# Swap-based param-reset vs. collisional amplitude-damping model

Loads the two `LCH_qc_swap_paramreset` runs (#485, #486) and overlays the measured
excited-state population vs. round with the **collisional amplitude-damping** theory
ported from `notebooks/calculator/collisional simulation of dynamics.ipynb` (cell 2).

Each round of the experiment iSWAPs the excited qubit's population into ancilla `q2`
and then resets `q2` — i.e. a partial swap into a freshly-reset ancilla that is then
discarded, exactly the collisional model. Tune `gamma`, `dt`, `P0` per run by eye
(`theta_per_round = sqrt(gamma*dt)` sets the decay; `P0` the initial excited population).

In [ ]:
# Cell 1 — setup & manual knobs
import os
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
import qutip as qt


def _find_repo_root(start):
    """Walk up from `start` until a directory containing the `scqat` package is found."""
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "scqat")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


# scqat is not pip-installed; add the repo root to sys.path so `scqat` resolves
# (same pattern as analysis/try_ramsey_new.py).
_ROOT = _find_repo_root(os.getcwd())
if _ROOT and _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from scqat.parsers import load_xarray_h5

_DATA_ROOT = r"D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\raw_data\2026-06-29"

# One entry per run. gamma / dt / P0 are the MANUAL KNOBS you tune by eye:
#   theta_per_round = sqrt(gamma * dt)  -> excited pop decays ~ cos^2(theta) per round
#   P0              = initial excited population (absorbs prep/readout infidelity)
RUNS = [
    {"path": os.path.join(_DATA_ROOT, "#485_LCH_qc_swap_paramreset_174040"),
     "gamma": 1, "dt": 1, "P0": 0.83},
    {"path": os.path.join(_DATA_ROOT, "#486_LCH_qc_swap_paramreset_174110"),
     "gamma": 1, "dt": 1, "P0": 0.83},
]

In [ ]:
# Cell 2 — load experiment + marginal population
def excite_qubit_of(run_dir):
    """Read the run's node.json and return the qubit that was excited (x180)."""
    with open(os.path.join(run_dir, "node.json"), "r", encoding="utf-8") as f:
        node = json.load(f)
    return node["data"]["parameters"]["model"]["excite_qubit"]


def experimental_curve(run_dir):
    """Return (rounds, P_excited, qubit_name) for the run's excited qubit.

    P_excited(round) = fraction of shots the excited qubit was measured in |1>, i.e. the
    per-qubit marginal population — mirrors marginal_populations in LCHQMDriver
    customized/node/_qc_populations.py: (state >= 1).mean("shot").
    """
    q = excite_qubit_of(run_dir)
    ds = load_xarray_h5(os.path.join(run_dir, "ds_raw.h5"))
    rounds = ds["round"].values
    p = (ds["state"].sel(qubit=q) >= 1).mean("shot").values
    return rounds, p, q

In [ ]:
# Cell 3 — collisional amplitude-damping theory (port of the reference notebook cell 2)
def collisional_excited_population(rounds, gamma, dt, P0=1.0):
    """Excited-state population vs round from the collisional amplitude-damping circuit.

    Faithful port of `collisional simulation of dynamics.ipynb` cell 2: each round the
    system qubit is partial-swapped (U = expm(-1j*sqrt(gamma*dt)*H), H = sp⊗sm + sm⊗sp)
    into a fresh ground-state ancilla which is then discarded (ptrace). Parameterized so
    it can be evaluated at the experiment's integer rounds; the initial excited
    population is scaled by P0 (prep/readout infidelity).
    """
    sp, sm = qt.sigmap(), qt.sigmam()
    H = qt.tensor(sp, sm) + qt.tensor(sm, sp)
    U = qt.Qobj.expm(-1j * np.sqrt(gamma * dt) * H)

    ket_e, ket_g = qt.basis(2, 0), qt.basis(2, 1)  # |e> = basis(2,0): rho[0,0] is excited pop
    rho_g = qt.ket2dm(ket_g)
    rho_sys = P0 * qt.ket2dm(ket_e) + (1.0 - P0) * rho_g

    n_max = int(np.max(rounds))
    pops = [np.real(rho_sys[0, 0])]
    for _ in range(n_max):
        rho_sys = (U * qt.tensor(rho_sys, rho_g) * U.dag()).ptrace(0)
        pops.append(np.real(rho_sys[0, 0]))
    pops = np.array(pops)
    return pops[np.asarray(rounds, dtype=int)]

In [ ]:
# Cell 4 — one independent figure per run: experiment + collisional theory
for run in RUNS:
    rounds, p_exp, q = experimental_curve(run["path"])
    gamma, dt, P0 = run["gamma"], run["dt"], run["P0"]
    p_th = collisional_excited_population(rounds, gamma, dt, P0)

    run_id = os.path.basename(run["path"])
    theta = float(np.sqrt(gamma * dt))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(rounds, p_exp, "o", label=f"experiment ({q})")
    ax.plot(rounds, p_th, "-", label="collisional theory")
    # continuous reference: the notebook's exp(-gamma*t) overlay, using the collisional
    # model's physical time t = n*dt (the notebook's tlist) — not the bare round index n.
    t = np.asarray(rounds) * dt
    ax.plot(rounds, P0 * np.exp(-gamma * t), "--", color="grey",
            alpha=0.6, label=r"$P_0\,e^{-\gamma t},\ t=n\,dt$")
    ax.set_xlabel("Number of swap+reset rounds")
    ax.set_ylabel("P(excited)")
    ax.set_title(
        f"{run_id}\n"
        f"excited={q}   γ={gamma}, dt={dt}, P0={P0}   "
        f"(θ/round={theta:.3f}, cos²θ={np.cos(theta) ** 2:.3f})"
    )
    ax.legend()
    fig.tight_layout()

plt.show()